In [ ]:
# Base oficial Default of Credit Card Clients, com 30.000 registros, 23 variáveis preditoras e licença CC BY 4.0.
%pip install ucimlrepo

In [ ]:
# PASSO 1 - Carregamento e visualização inicial do dataset UCI

import pandas as pd
from ucimlrepo import fetch_ucirepo

# Carregar dataset oficial UCI
dataset = fetch_ucirepo(id=350)

# Variáveis e classificação de inadimplência
X = dataset.data.features
y = dataset.data.targets

# Reunir em um DataFrame
df = pd.concat([X, y], axis=1)

assert len(df) == 30000
from pathlib import Path
raw_path = Path("data/raw/default_credit_clients.csv")
raw_path.parent.mkdir(parents=True, exist_ok=True)
if not raw_path.exists():
    df.to_csv(raw_path, index=False)

print("Total de linhas:", df.shape[0])
print("Total de colunas:", df.shape[1])

display(df.head())


### 📝 Observação 01 — Carregamento e Estrutura dos Dados

Execute as células de código para produzir os resultados desta etapa.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# PASSO 2 - Análise exploratória inicial
print("=== INFORMAÇÕES DO DATASET ===")
df.info()

print("\n=== VALORES AUSENTES ===")
print(df.isnull().sum())

print("\n=== ESTATÍSTICAS DESCRITIVAS ===")
display(df.describe())

### 📝 Observação 02 — Análise Exploratória dos Dados (EDA)

analisar a estrutura, os tipos de dados e as
características estatísticas do dataset.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# PASSO 3 - Qualidade dos dados

print("=== QUALIDADE DO DATASET ===")

print("Total de registros:", len(df))
print("Total de colunas:", df.shape[1])

print("\nValores ausentes:")
print(df.isnull().sum().sum())

print("\nRegistros duplicados:")
print(df.duplicated().sum())

print("\nTipos de dados:")
print(df.dtypes.value_counts())

### 📝 Observação 03 — Verificação da Qualidade dos Dados

identificar valores ausentes, registros duplicados
e tipos de dados antes do treinamento do modelo.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# PASSO 4 - Distribuição dos clientes inadimplentes

import matplotlib.pyplot as plt

contagem = df["Y"].value_counts().sort_index()
percentual = df["Y"].value_counts(
    normalize=True
).sort_index() * 100

print("=== DISTRIBUIÇÃO DA INADIMPLÊNCIA ===")
print("\nQuantidade de clientes:")
print(contagem)

print("\nPercentual de clientes:")
print(percentual.round(2))

plt.figure(figsize=(8, 5))

barras = plt.bar(
    ["Não inadimplente", "Inadimplente"],
    contagem.values
)

plt.bar_label(barras)

plt.title("Distribuição de Inadimplência")
plt.ylabel("Quantidade de clientes")
plt.xlabel("Situação do cliente")

plt.show()

### 📝 Observação 04 — Distribuição da Inadimplência

analisar a proporção de clientes inadimplentes
e não inadimplentes no dataset.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# PASSO 5 - Identificação de inconsistências categóricas

colunas = {
    "X2": "Sexo",
    "X3": "Escolaridade",
    "X4": "Estado civil"
}

print("=== ANÁLISE DAS VARIÁVEIS CATEGÓRICAS ===")

for coluna, descricao in colunas.items():

    print(f"\n{coluna} - {descricao}")

    frequencia = df[coluna].value_counts().sort_index()

    percentual = (
        df[coluna].value_counts(normalize=True)
        .sort_index() * 100
    )

    resultado = pd.DataFrame({
        "Quantidade": frequencia,
        "Percentual (%)": percentual.round(2)
    })

    display(resultado)

### 📝 Observação 05 — Análise das Variáveis Categóricas

verificar a distribuição dos códigos utilizados
nas variáveis Sexo (X2), Escolaridade (X3) e Estado Civil (X4).

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# PASSO 6 - Tratamento das variáveis categóricas

# Criar cópia para preservar os dados originais
df_limpo = df.copy()

# X3 - Escolaridade
# Agrupar códigos não documentados em Outros
df_limpo["X3"] = df_limpo["X3"].replace({
    0: 4,
    5: 4,
    6: 4
})

# X4 - Estado civil
# Agrupar código desconhecido em Outros
df_limpo["X4"] = df_limpo["X4"].replace({
    0: 3
})

print("=== RESULTADO DO TRATAMENTO ===")

print("\nEscolaridade (X3):")
print(df_limpo["X3"].value_counts().sort_index())

print("\nEstado civil (X4):")
print(df_limpo["X4"].value_counts().sort_index())

print("\nTotal de registros:", len(df_limpo))

### 📝 Observação 06 — Padronização das Variáveis Categóricas

corrigir códigos categóricos não documentados
e padronizar os dados para o treinamento do modelo.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# PASSO 7 - Correlação das variáveis com inadimplência

import pandas as pd
import matplotlib.pyplot as plt

# Correlação de Pearson
correlacoes = df_limpo.corr(numeric_only=True)["Y"].drop("Y")

# Ordenar pelo valor absoluto da correlação
correlacoes = correlacoes.reindex(
    correlacoes.abs().sort_values(ascending=False).index
)

print("=== CORRELAÇÃO COM INADIMPLÊNCIA ===")
display(correlacoes.to_frame(name="Correlação com Y"))

# Gráfico
plt.figure(figsize=(10, 8))

correlacoes.sort_values().plot(kind="barh")

plt.title("Correlação de Pearson com Inadimplência")
plt.xlabel("Coeficiente de correlação")
plt.ylabel("Variáveis")
plt.axvline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

### 📝 Observação 07 — Correlação das Variáveis com a Inadimplência

identificar quais variáveis apresentam maior
associação linear com a inadimplência (Y).

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# PASSO 8 - Análise de outliers financeiros

import matplotlib.pyplot as plt

colunas = ["X1", "X12"]

print("=== ANÁLISE DE VALORES EXTREMOS ===")

display(
    df_limpo[colunas].describe().T
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, coluna in zip(axes, colunas):
    ax.boxplot(df_limpo[coluna].dropna())
    ax.set_title(f"Distribuição de {coluna}")
    ax.set_ylabel("Valor em NT$")
    ax.set_xticks([1])
    ax.set_xticklabels([coluna])

plt.tight_layout()
plt.show()

### 📝 Observação 08 — Análise de Valores Extremos (Outliers)

investigar a distribuição dos valores financeiros
e identificar possíveis valores extremos no dataset.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# PASSO 9 - Identificação de outliers pelo método IQR

colunas = ["X1", "X12"]

resultados = []

for coluna in colunas:
    Q1 = df_limpo[coluna].quantile(0.25)
    Q3 = df_limpo[coluna].quantile(0.75)

    IQR = Q3 - Q1

    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    outliers = (
        (df_limpo[coluna] < limite_inferior) |
        (df_limpo[coluna] > limite_superior)
    )

    quantidade = outliers.sum()
    percentual = quantidade / len(df_limpo) * 100

    resultados.append({
        "Variável": coluna,
        "Limite inferior": limite_inferior,
        "Limite superior": limite_superior,
        "Outliers": quantidade,
        "Percentual (%)": round(percentual, 2)
    })

resultado_iqr = pd.DataFrame(resultados)

print("=== IDENTIFICAÇÃO DE OUTLIERS - IQR ===")
display(resultado_iqr)

### 📝 Observação 09 — Identificação de Outliers pelo Método IQR

identificar e quantificar valores extremos
nas variáveis financeiras utilizando o intervalo
interquartil (IQR).

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# PASSO 10 - Separação dos dados para treinamento e teste

from app.data_split import dividir_dados

# Separar variáveis preditoras e variável-alvo
X = df_limpo.drop(columns=["Y"])
y = df_limpo["Y"]

# Divisão estratificada por grupos: aproximadamente 80%/20%
X_train, X_test, y_train, y_test = dividir_dados(X, y)

print("=== DIVISÃO DOS DADOS ===")

print("Treinamento:", X_train.shape)
print("Teste:", X_test.shape)

print("\nDistribuição da inadimplência no treinamento:")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nDistribuição da inadimplência no teste:")
print(y_test.value_counts(normalize=True).mul(100).round(2))

### 📝 Observação 10 — Separação dos Dados de Treinamento e Teste

dividir o dataset em conjuntos independentes
para treinamento e avaliação do modelo de Machine Learning.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.


In [ ]:
# PASSO 11 - Exportação dos dados preparados

from pathlib import Path

# Localizar a pasta principal do projeto
projeto = Path.cwd()
if projeto.name == "notebooks":
    projeto = projeto.parent

# Criar diretório de saída
pasta_saida = projeto / "data" / "processed"
pasta_saida.mkdir(parents=True, exist_ok=True)

# Salvar os dados preparados
arquivo = pasta_saida / "dataset_limpo.csv"

df_limpo.to_csv(arquivo, index=False)

print("=== EXPORTAÇÃO CONCLUÍDA ===")
print("Arquivo:", arquivo)
print("Total de registros:", len(df_limpo))
print("Total de colunas:", df_limpo.shape[1])

### 📝 Observação 11 — Exportação dos Dados Preparados

salvar os dados preparados para utilização
nas próximas etapas do projeto de Machine Learning.

Os resultados numéricos são gerados durante a execução. Consulte o README e `docs/metricas-verificadas.json` para a avaliação auditada com separação por grupos.
